In [1]:
# ✅ SMARTINVEST — Feature Engineering Notebook (version propre et restructurée)

# === 1. Imports ===
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import os, csv, sys
csv.field_size_limit(sys.maxsize)

# === 2. Charger le fichier principal ===
df = pd.read_csv("df_streamlit.csv")

# === 3. Fonction : calcul de distance Haversine en km ===
def haversine_min_distance(df_base, df_poi,
    base_lat='latitude', base_lon='longitude',
    poi_lat='lat', poi_lon='lon',
    label='dist_min_km'):

    R = 6371.0  # rayon terrestre en km
    poi = df_poi[[poi_lat, poi_lon]].dropna()
    lat2 = np.radians(poi[poi_lat].values)
    lon2 = np.radians(poi[poi_lon].values)

    def min_dist(row):
        if pd.isna(row[base_lat]) or pd.isna(row[base_lon]):
            return np.nan
        a1, o1 = np.radians(row[[base_lat, base_lon]].astype(float))
        dlat = lat2 - a1
        dlon = lon2 - o1
        a = np.sin(dlat/2)**2 + np.cos(a1)*np.cos(lat2) * np.sin(dlon/2)**2
        d = 2 * R * np.arcsin(np.sqrt(a))
        return d.min()

    df_base[label] = df_base.apply(min_dist, axis=1)
    return df_base

# === 4. DISTANCE AUX COMMERCES ===
df_shop = pd.read_csv("datashop.csv", sep=';', encoding='utf-8', engine='python')
df_shop = df_shop[df_shop['com_insee'].astype(str).str.startswith('75')]
df_shop['lat'] = df_shop['Y'].astype(float)
df_shop['lon'] = df_shop['X'].astype(float)
df_shop = df_shop[['lat', 'lon']].dropna()
df = haversine_min_distance(df, df_shop, label='dist_commerce_km')

# === 5. DISTANCE AUX ESPACES VERTS ===
df_green = pd.read_csv("espaces_verts.csv", sep=';')
df_green[['lon', 'lat']] = df_green['geom_x_y'].str.split(',', expand=True).astype(float)
df_green = df_green[['lat', 'lon']].dropna()
df = haversine_min_distance(df, df_green, label='dist_espaces_verts_km')

# === 6. DISTANCE AUX GARES ===
df_gares = pd.read_csv("emplacement-des-gares-idf.csv", sep=';', encoding='utf-8')
df_gares.columns = df_gares.columns.str.strip()
df_gares[['lat', 'lon']] = df_gares['Geo Point'].str.split(',', expand=True).astype(float)
df_gares = df_gares[['lat', 'lon']].dropna()
df = haversine_min_distance(df, df_gares, label='dist_gare_km')

# === 7. DISTANCE AUX ÉCOLES PRIMAIRES ===
df_ecoles = pd.read_csv("secteurs-scolaires-ecoles-elementaires.csv", sep=';', encoding='utf-8')
df_ecoles.columns = df_ecoles.columns.str.strip()
if 'geo_point_2d' in df_ecoles.columns:
    df_ecoles[['lat', 'lon']] = df_ecoles['geo_point_2d'].str.split(',', expand=True).astype(float)
    df_ecoles = df_ecoles[['lat', 'lon']].dropna()
    df = haversine_min_distance(df, df_ecoles, label='dist_ecole_primaire_km')
else:
    print("Colonne 'geo_point_2d' manquante pour les écoles.")

# === 8. DISTANCE AUX COLLÈGES ===
df_colleges = pd.read_csv("etablissements-scolaires-colleges.csv", sep=';', encoding='utf-8')
df_colleges.columns = df_colleges.columns.str.strip()
if 'geo_point_2d' in df_colleges.columns:
    df_colleges[['lat', 'lon']] = df_colleges['geo_point_2d'].str.split(',', expand=True).astype(float)
    df_colleges = df_colleges[['lat', 'lon']].dropna()
    df = haversine_min_distance(df, df_colleges, label='dist_college_km')
else:
    print("Colonne 'geo_point_2d' manquante pour les collèges.")

# === 9. DISTANCE AUX UNIVERSITÉS ===
df_univ = pd.read_csv("fr-esr-principaux-etablissements-enseignement-superieur.csv", sep=';')
df_paris_univ = df_univ[df_univ['localisation'].str.contains("Paris", na=False)].copy()
df_paris_univ[['lat', 'lon']] = df_paris_univ['géolocalisation'].str.split(',', expand=True).astype(float)
df_paris_univ = df_paris_univ[['lat', 'lon']].dropna()
df = haversine_min_distance(df, df_paris_univ, label='dist_universite_km')

# === 10. Sauvegarde finale ===
df.to_csv("df_ready.csv", index=False)
print("\n✅ Fichier enrichi enregistré sous df_ready.csv avec toutes les distances en km")



✅ Fichier enrichi enregistré sous df_ready.csv avec toutes les distances en km


In [2]:
df = pd.read_csv("df_ready.csv")

In [3]:
df.columns

Index(['valeur_fonciere', 'surface_reelle_bati', 'nombre_pieces_principales',
       'longitude', 'latitude', 'prix_m2', 'jour', 'mois', 'annee',
       'arrondissement', 'dist_commerce_km', 'dist_espaces_verts_km',
       'dist_gare_km', 'dist_ecole_primaire_km', 'dist_college_km',
       'dist_universite_km'],
      dtype='object')

In [4]:
df.head()

,valeur_fonciere,surface_reelle_bati,nombre_pieces_principales,longitude,latitude,prix_m2,jour,mois,annee,arrondissement,dist_commerce_km,dist_espaces_verts_km,dist_gare_km,dist_ecole_primaire_km,dist_college_km,dist_universite_km
0,562400.0,55.0,2.0,2.251902,48.838420,10225.454545,5,1,2024,16,0.081620,6786.078418,0.385445,0.645159,0.453587,2.678869
1,580000.0,78.0,3.0,2.251124,48.843025,7435.897436,15,1,2024,16,0.064123,6786.411745,0.624619,0.313463,0.302854,2.778257
2,880000.0,85.0,4.0,2.253974,48.835999,10352.941176,19,1,2024,16,0.197244,6785.802236,0.317329,0.662126,0.668234,2.632812
3,252165.0,34.0,1.0,2.281940,48.830593,7416.617647,20,3,2024,15,0.200587,6783.768885,0.127436,0.402837,0.951336,2.354333
4,485000.0,89.0,4.0,2.252871,48.837354,5449.438202,5,4,2024,16,0.155641,6785.953373,0.318380,0.699537,0.535693,2.657511


In [5]:
import pandas as pd
import numpy as np

# Fonction Haversine
def haversine_min_distance(df_base, df_poi,
    base_lat='latitude', base_lon='longitude',
    poi_lat='lat', poi_lon='lon',
    label='dist_espaces_verts_km'):

    R = 6371.0
    poi = df_poi[[poi_lat, poi_lon]].dropna()
    lat2 = np.radians(poi[poi_lat].values)
    lon2 = np.radians(poi[poi_lon].values)

    def min_dist(row):
        if pd.isna(row[base_lat]) or pd.isna(row[base_lon]):
            return np.nan
        a1, o1 = np.radians(row[[base_lat, base_lon]].astype(float))
        dlat = lat2 - a1
        dlon = lon2 - o1
        a = np.sin(dlat/2)**2 + np.cos(a1)*np.cos(lat2) * np.sin(dlon/2)**2
        d = 2 * R * np.arcsin(np.sqrt(a))
        return d.min()

    df_base[label] = df_base.apply(min_dist, axis=1)
    return df_base

# Charger les fichiers
df = pd.read_csv("df_streamlit.csv")  # ou df_ready.csv
verts = pd.read_csv("espaces_verts.csv", sep=';')

# Extraire lat/lon depuis 'geom_x_y'
verts[['lon', 'lat']] = verts['geom_x_y'].str.split(',', expand=True).astype(float)
verts = verts[['lat', 'lon']].dropna()

# Calcul distance
df = haversine_min_distance(df, verts, label='dist_espaces_verts_km')

# Sauvegarde
df.to_csv("df_with_espaces_verts.csv", index=False)
print("✅ distances aux espaces verts mises à jour et enregistrées dans df_with_espaces_verts.csv")


✅ distances aux espaces verts mises à jour et enregistrées dans df_with_espaces_verts.csv


In [6]:
import pandas as pd

# Charger le fichier complet avec toutes les features
df_ready = pd.read_csv("df_ready.csv")

# Charger uniquement la colonne corrigée
df_corrected = pd.read_csv("df_with_espaces_verts.csv")

# Supprimer l'ancienne colonne erronée
if 'dist_espaces_verts_km' in df_ready.columns:
    df_ready = df_ready.drop(columns=['dist_espaces_verts_km'])

# Ajouter la colonne corrigée (en s’assurant du même ordre)
df_ready['dist_espaces_verts_km'] = df_corrected['dist_espaces_verts_km']

# Sauvegarde du nouveau fichier propre
df_ready.to_csv("df_ready_clean.csv", index=False)
print("✅ Nouveau fichier enregistré : df_ready_clean.csv (avec espaces verts corrigé)")


✅ Nouveau fichier enregistré : df_ready_clean.csv (avec espaces verts corrigé)


In [7]:
df = pd.read_csv("df_ready_clean.csv")

In [8]:
df.head()

,valeur_fonciere,surface_reelle_bati,nombre_pieces_principales,longitude,latitude,prix_m2,jour,mois,annee,arrondissement,dist_commerce_km,dist_gare_km,dist_ecole_primaire_km,dist_college_km,dist_universite_km,dist_espaces_verts_km
0,562400.0,55.0,2.0,2.251902,48.838420,10225.454545,5,1,2024,16,0.081620,0.385445,0.645159,0.453587,2.678869,6786.078418
1,580000.0,78.0,3.0,2.251124,48.843025,7435.897436,15,1,2024,16,0.064123,0.624619,0.313463,0.302854,2.778257,6786.411745
2,880000.0,85.0,4.0,2.253974,48.835999,10352.941176,19,1,2024,16,0.197244,0.317329,0.662126,0.668234,2.632812,6785.802236
3,252165.0,34.0,1.0,2.281940,48.830593,7416.617647,20,3,2024,15,0.200587,0.127436,0.402837,0.951336,2.354333,6783.768885
4,485000.0,89.0,4.0,2.252871,48.837354,5449.438202,5,4,2024,16,0.155641,0.318380,0.699537,0.535693,2.657511,6785.953373


In [38]:
# === Espaces verts via Open Data Paris (Geo point formaté) ===

import pandas as pd

# Chargement du fichier (modifie le nom selon ton cas)
df_green = pd.read_csv("espaces_verts.csv", sep=';')

# Séparation des coordonnées (attention : lat,lon dans Geo point)
df_green[['lat', 'lon']] = df_green['Geo point'].str.split(',', expand=True).astype(float)

# Vérif rapide
print(df_green[['lat', 'lon']].head())
print(df_green[['lat', 'lon']].describe())

# Filtrage Paris uniquement (sécurité)
df_green = df_green[
    (df_green['lat'] >= 48.80) & (df_green['lat'] <= 48.92) &
    (df_green['lon'] >= 2.25) & (df_green['lon'] <= 2.42)
].copy()

print("Espaces verts conservés :", len(df_green))



         lat       lon
0  48.843250  2.413392
1  48.887385  2.325058
2  48.837096  2.390989
3  48.843455  2.326865
4  48.845658  2.352887
               lat          lon
count  2440.000000  2440.000000
mean     48.858876     2.348542
std       0.023043     0.041222
min      48.760877     2.233582
25%      48.839366     2.319882
50%      48.857888     2.353354
75%      48.878418     2.380083
max      48.908150     2.466529
Espaces verts conservés : 2423


In [39]:
df = haversine_min_distance_fast(df, df_green, label='dist_espaces_verts_km')
print(df['dist_espaces_verts_km'].describe())


count    23150.000000
mean         0.117378
std          0.067623
min          0.006286
25%          0.067327
50%          0.104948
75%          0.153538
max          0.495980
Name: dist_espaces_verts_km, dtype: float64


In [40]:
from sklearn.neighbors import BallTree
import numpy as np

def haversine_min_distance_fast(df_main, df_poi, col_lat_main, col_lon_main, col_lat_poi, col_lon_poi, label='distance_km'):
    # Conversion en radians
    main_coords = np.radians(df_main[[col_lat_main, col_lon_main]])
    poi_coords = np.radians(df_poi[[col_lat_poi, col_lon_poi]])
    
    # Création de l’arbre
    tree = BallTree(poi_coords, metric='haversine')
    
    # Recherche plus proche voisin
    distances, _ = tree.query(main_coords, k=1)
    df_main[label] = distances.flatten() * 6371  # conversion en km
    return df_main


In [41]:
# === ESPACES VERTS (déjà nettoyé) ===
df = haversine_min_distance_fast(df, df_green, 
                                 col_lat_main='latitude', col_lon_main='longitude',
                                 col_lat_poi='lat', col_lon_poi='lon',
                                 label='dist_espaces_verts_km')


In [42]:
# === COMMERCES ===
df = haversine_min_distance_fast(df, df_shop, 
                                 col_lat_main='latitude', col_lon_main='longitude',
                                 col_lat_poi='lat', col_lon_poi='lon',
                                 label='dist_commerce_km')

# === GARES ===
df = haversine_min_distance_fast(df, df_gares, 
                                 col_lat_main='latitude', col_lon_main='longitude',
                                 col_lat_poi='lat', col_lon_poi='lon',
                                 label='dist_gare_km')

# === ÉCOLES ===
df = haversine_min_distance_fast(df, df_ecoles, 
                                 col_lat_main='latitude', col_lon_main='longitude',
                                 col_lat_poi='lat', col_lon_poi='lon',
                                 label='dist_ecole_km')


In [43]:
# Appel groupé
df = haversine_min_distance_fast(df, df_green, 'latitude', 'longitude', 'lat', 'lon', 'dist_espaces_verts_km')
df = haversine_min_distance_fast(df, df_shop, 'latitude', 'longitude', 'lat', 'lon', 'dist_commerce_km')
df = haversine_min_distance_fast(df, df_gares, 'latitude', 'longitude', 'lat', 'lon', 'dist_gare_km')
df = haversine_min_distance_fast(df, df_ecoles, 'latitude', 'longitude', 'lat', 'lon', 'dist_ecole_km')
